In [1]:
import requests
from azure.identity import AzureCliCredential

In [2]:
cred = AzureCliCredential()
token = cred.get_token("https://graph.microsoft.com/.default").token
H = {"Authorization": f"Bearer {token}"}
GRAPH = "https://graph.microsoft.com/v1.0"

In [4]:
host = "myacu.sharepoint.com"
site_path = "/sites/SquareEyes"
site = requests.get(f"{GRAPH}/sites/{host}:{site_path}", headers=H).json()
site_id = site["id"]

In [9]:
drives = requests.get(f"{GRAPH}/sites/{site_id}/drives", headers=H).json()["value"]

for d in drives:
    print(d["name"], "->", d["id"])

Square_Eyes_DP20_Data -> b!RwAwqeSjb02ZBzCucnQzb7p8kMSgzShAgnfcHlYRM5ekVNAtFcg0T7rLco_8Ak8X
Documents -> b!RwAwqeSjb02ZBzCucnQzb7p8kMSgzShAgnfcHlYRM5efY57B4vX3SImppFHrQsPb


In [ ]:
target = "Square_Eyes_DP20_Data"  # whatever the library is called in SharePoint
drive_id = next(d["id"] for d in drives if d["name"] == target)

In [ ]:
from itertools import islice


def iter_files(drive_id, start_path=""):
    """Recursively yield (path, item) for every file under start_path."""
    # Resolve the starting folder to an item id (root if no path given)
    if start_path:
        root = requests.get(
            f"{GRAPH}/drives/{drive_id}/root:/{start_path}", headers=H
        ).json()
    else:
        root = requests.get(f"{GRAPH}/drives/{drive_id}/root", headers=H).json()

    stack = [(root["id"], start_path)]  # (folder id, path so far)
    while stack:
        folder_id, prefix = stack.pop()
        url = f"{GRAPH}/drives/{drive_id}/items/{folder_id}/children?$top=200"
        while url:  # inner loop handles pagination
            page = requests.get(url, headers=H).json()
            for item in page.get("value", []):
                path = f"{prefix}/{item['name']}" if prefix else item["name"]
                if "folder" in item:
                    stack.append((item["id"], path))  # descend later
                else:
                    yield path, item
            url = page.get("@odata.nextLink")

In [ ]:
def fetch_csv(item_id):
    r = requests.get(f"{GRAPH}/drives/{drive_id}/items/{item_id}/content", headers=H)
    r.raise_for_status()
    return r.content

In [ ]:
import io
import pandas as pd

DATETIME_COL = "DateTime"  # set to your actual column name
SENTINEL = "9999-12-31 23:59:59"


def fix_missing(raw: bytes, col=DATETIME_COL, sentinel=SENTINEL):
    # dtype=str + keep_default_na=False -> every cell stays an exact string,
    # nothing gets reformatted or turned into NaN behind your back.
    df = pd.read_csv(io.BytesIO(raw), dtype=str, keep_default_na=False)

    # "missing" = empty or whitespace-only. Add tokens here if your data
    # uses literal placeholders, e.g. .isin(["", "NA", "NULL"]).
    mask = df[col].str.strip() == ""
    n = int(mask.sum())
    df.loc[mask, col] = sentinel

    out = io.StringIO()
    df.to_csv(out, index=False)
    return out.getvalue().encode("utf-8"), n

In [18]:
def upload_csv(item_id, data: bytes):
    r = requests.put(
        f"{GRAPH}/drives/{drive_id}/items/{item_id}/content",
        headers={**H, "Content-Type": "text/csv"},
        data=data,
    )
    r.raise_for_status()
    return r.json()

In [46]:
def find_csvs(drive_id, start_path):
    url = f"{GRAPH}/drives/{drive_id}/root:/{start_path}:/search(q='.csv')"
    while url:
        page = requests.get(url, headers=H).json()
        for item in page.get("value", []):
            if "file" not in item or item["name"].lower() != "image data import.csv":
                continue
            # search() drops parentReference.path, so fetch full metadata for a clean path
            full = requests.get(
                f"{GRAPH}/drives/{drive_id}/items/{item['id']}"
                "?$select=id,name,parentReference",
                headers=H,
            ).json()
            parent = full.get("parentReference", {}).get("path", "")
            folder = parent.split("root:", 1)[-1].lstrip("/")  # path within the library
            path = f"{folder}/{item['name']}" if folder else item["name"]
            yield path, item
        url = page.get("@odata.nextLink")

In [50]:
csvs = (
    (path, item)
    for path, item in find_csvs(
        drive_id, "Participant_Data/Main Study/Community Sample"
    )
)

fixed_files = 0
skipped = 0
errors = []

for path, item in csvs:
    try:
        raw = fetch_csv(item["id"])
        fixed, changed = fix_missing(raw)
        if changed > 0:
            upload_csv(item["id"], fixed)
            fixed_files += 1
            print(f"✓ Fixed {changed} missing value(s) in {path}")
        else:
            skipped += 1
            print(f"– No changes needed: {path}")
    except Exception as e:
        errors.append((path, e))
        print(f"✗ ERROR on {path}: {e}")

print(
    f"\nDone. {fixed_files} file(s) updated, {skipped} unchanged, {len(errors)} error(s)."
)
if errors:
    for path, e in errors:
        print(f"  {path}: {e}")


– No changes needed: Participant_Data/Main Study/Community Sample/Practice Data/Image Data Import.csv
✓ Fixed 5 missing value(s) in Participant_Data/Main Study/Community Sample/4330/Baseline/Images/Image Data Import.csv
✓ Fixed 3 missing value(s) in Participant_Data/Main Study/Community Sample/1614/Time_1/Images/Image Data Import.csv
✓ Fixed 74 missing value(s) in Participant_Data/Main Study/Community Sample/4346/Baseline/Images/Image Data Import.csv
✓ Fixed 4 missing value(s) in Participant_Data/Main Study/Community Sample/1281/Time_2/Images/Image Data Import.csv
✓ Fixed 993 missing value(s) in Participant_Data/Main Study/Community Sample/1063/Time_2/Images/Image Data Import.csv
– No changes needed: Participant_Data/Main Study/Community Sample/1341/Time 2/Images/Image Data Import.csv
✓ Fixed 287 missing value(s) in Participant_Data/Main Study/Community Sample/1542/Time_1/Images/Image Data Import.csv
✓ Fixed 958 missing value(s) in Participant_Data/Main Study/Community Sample/1605/Base